# Cosmos 3 Nano Reasoner — full MMAD benchmark (server)

Runs all **39,670 MMAD questions over 8,366 unique industrial images** with the official `nvidia/Cosmos3-Nano` Reasoner in BF16. Ground truth is never included in model prompts. Predictions are appended immediately and evaluation joins labels only after inference.

Recommended: Linux, CUDA GPU with BF16 support (A100/H100/RTX 6000 Ada or stronger), at least 40 GiB VRAM and 55 GiB free disk. Set `MMAD_WORKDIR`, `MMAD_BATCH_SIZE`, or `MMAD_MIRROR_DIR` before Run All if required. Re-running resumes from `predictions.jsonl`.


In [ ]:
%pip install -q -U "transformers>=5.14.0" accelerate qwen-vl-utils safetensors remotezip requests pillow pandas matplotlib seaborn


In [ ]:
import os, sys, json, time, shutil, signal, subprocess
from pathlib import Path

default_work = '/kaggle/working/cosmos3_mmad_full' if Path('/kaggle/working').exists() else '/workspace/cosmos3_mmad_full'
WORK = Path(os.environ.get('MMAD_WORKDIR', default_work)).resolve()
REPO = WORK / 'mini-world-model'
DATA = WORK / 'data'
CACHE = WORK / 'archive_cache'
OUT = WORK / 'outputs'
HF_CACHE = WORK / 'hf_cache'
MIRROR_TEXT = os.environ.get('MMAD_MIRROR_DIR', '').strip()
MIRROR = Path(MIRROR_TEXT).resolve() if MIRROR_TEXT else None
for folder in (WORK, DATA, CACHE, OUT, HF_CACHE): folder.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

REPO_URL = 'https://github.com/anhsown/mini-world-model.git'
if REPO.exists(): subprocess.run(['git','-C',str(REPO),'pull','--ff-only'], check=True)
else: subprocess.run(['git','clone','--depth','1',REPO_URL,str(REPO)], check=True)
BASE = REPO / 'research/mmad_model_benchmark'
sys.path.insert(0, str(BASE))
print('workdir:', WORK)
print('repo commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())


In [ ]:
import torch, transformers
assert torch.cuda.is_available(), 'CUDA GPU is required.'
gpus=[]
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i)
    gpus.append({'index':i,'name':p.name,'vram_gib':round(p.total_memory/2**30,2)})
free_gib=shutil.disk_usage(WORK).free/2**30
preflight={'torch':torch.__version__,'transformers':transformers.__version__,'bf16_supported':torch.cuda.is_bf16_supported(),'free_disk_gib':round(free_gib,2),'gpus':gpus}
print(json.dumps(preflight,indent=2))
assert preflight['bf16_supported'], 'Official Cosmos 3 precision is BF16; use A100/H100/Ada-class hardware.'
assert free_gib >= 55, f'Need at least 55 GiB free disk; found {free_gib:.2f} GiB.'


In [ ]:
# Build the canonical full manifest and materialize only its 8,366 unique images.
subprocess.run([sys.executable,str(BASE/'prepare_full.py'),'--output',str(DATA),'--cache',str(CACHE),'--range-download'],cwd=BASE,check=True)
manifest_path=DATA/'full_manifest.json'
manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
missing=[r['image_file'] for r in manifest['records'] if not (DATA/r['image_file']).exists()]
assert len(manifest['records'])==39670, len(manifest['records'])
assert manifest['unique_images']==8366, manifest['unique_images']
assert not missing, f'{len(missing)} images missing'
print({'questions':len(manifest['records']),'unique_images':manifest['unique_images'],'sha256':manifest['manifest_sha256']})


In [ ]:
# Optional Hugging Face token (Kaggle secret HF_TOKEN or environment variable).
hf_token=os.environ.get('HF_TOKEN')
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token=UserSecretsClient().get_secret('HF_TOKEN')
    except Exception: hf_token=None
if hf_token:
    from huggingface_hub import login
    login(token=hf_token,add_to_git_credential=False)
print('HF authentication:', 'enabled' if hf_token else 'anonymous')


In [ ]:
# Official Cosmos 3 Nano unified checkpoint; Transformers instantiates only the Reasoner path.
from transformers import AutoProcessor, Cosmos3OmniForConditionalGeneration
MODEL_ID='nvidia/Cosmos3-Nano'
BATCH_SIZE=max(1,int(os.environ.get('MMAD_BATCH_SIZE','2')))
MAX_NEW_TOKENS=max(8,int(os.environ.get('MMAD_MAX_NEW_TOKENS','64')))
load_start=time.perf_counter()
processor=AutoProcessor.from_pretrained(MODEL_ID,min_pixels=256*28*28,max_pixels=512*28*28,token=hf_token)
model=Cosmos3OmniForConditionalGeneration.from_pretrained(MODEL_ID,dtype=torch.bfloat16,device_map='auto',low_cpu_mem_usage=True,attn_implementation='sdpa',token=hf_token).eval()
model.generation_config.do_sample=False
model.generation_config.temperature=None
model.generation_config.top_p=None
model.generation_config.top_k=None
run_config={'model':MODEL_ID,'precision':'official BF16 Reasoner','batch_size':BATCH_SIZE,'max_new_tokens':MAX_NEW_TOKENS,'manifest_sha256':manifest['manifest_sha256'],'load_seconds':round(time.perf_counter()-load_start,2),'gpu_info':gpus,'device_map':getattr(model,'hf_device_map',{})}
(OUT/'run_config.json').write_text(json.dumps(run_config,indent=2,default=str),encoding='utf-8')
print(json.dumps(run_config,indent=2,default=str))
for i in range(torch.cuda.device_count()): print(f'cuda:{i} allocated GiB',round(torch.cuda.memory_allocated(i)/2**30,2))


In [ ]:
from datetime import datetime, timezone
from qwen_vl_utils import process_vision_info
from common.mmad import SYSTEM_PROMPT,append_jsonl,evaluate_records,load_jsonl,parse_prediction,write_evaluation
PRED=OUT/'predictions.jsonl'
MIRROR_PRED=MIRROR/'predictions.jsonl' if MIRROR else None

def sync_mirror():
    if MIRROR_PRED and PRED.exists():
        MIRROR_PRED.parent.mkdir(parents=True,exist_ok=True)
        tmp=MIRROR_PRED.with_suffix('.jsonl.tmp'); shutil.copy2(PRED,tmp); tmp.replace(MIRROR_PRED)
def restore_mirror():
    if MIRROR_PRED and MIRROR_PRED.exists() and (not PRED.exists() or MIRROR_PRED.stat().st_size>PRED.stat().st_size):
        shutil.copy2(MIRROR_PRED,PRED)
restore_mirror()

def make_conversation(sample):
    return [{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':[{'type':'image','image':str((DATA/sample['image_file']).resolve())},{'type':'text','text':sample['prompt']}]}]

def infer_batch(batch):
    conversations=[make_conversation(s) for s in batch]
    texts=[processor.apply_chat_template(c,tokenize=False,add_generation_prompt=True) for c in conversations]
    image_inputs=[]
    for c in conversations:
        images,_=process_vision_info(c); image_inputs.append(images[0])
    inputs=processor(text=texts,images=image_inputs,padding=True,return_tensors='pt').to(model.device)
    torch.cuda.synchronize(); started=time.perf_counter()
    with torch.inference_mode(): generated=model.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=False)
    torch.cuda.synchronize(); elapsed=time.perf_counter()-started
    trimmed=[out[len(inp):] for inp,out in zip(inputs.input_ids,generated)]
    raws=processor.batch_decode(trimmed,skip_special_tokens=True,clean_up_tokenization_spaces=False)
    return [r.strip() for r in raws],elapsed/len(batch)

def emergency(*_): sync_mirror(); raise KeyboardInterrupt
signal.signal(signal.SIGTERM,emergency); signal.signal(signal.SIGINT,emergency)


In [ ]:
# Three-sample gate before committing to the full run.
smoke=[manifest['records'][i] for i in (0,5670,34000)]
raws,latency=infer_batch(smoke)
for sample,raw in zip(smoke,raws):
    print(sample['sample_id'],parse_prediction(raw),repr(raw[:300]),f'{latency:.2f}s')
assert any(parse_prediction(raw) for raw in raws),'No parseable smoke prediction; stop and inspect outputs.'


In [ ]:
# Full resumable inference. Each answer is written immediately; labels are not written here.
previous=load_jsonl(PRED)
done={r['sample_id'] for r in previous if r.get('status') in {'ok','parse_failure'}}
pending=[r for r in manifest['records'] if r['sample_id'] not in done]
print(f'total={len(manifest["records"])} done={len(done)} pending={len(pending)} batch={BATCH_SIZE}',flush=True)
cursor=0; active_batch=BATCH_SIZE; completed=0; started_all=time.perf_counter()
try:
    while cursor<len(pending):
        batch=pending[cursor:cursor+active_batch]
        try:
            raws,per_latency=infer_batch(batch)
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            if active_batch==1: raise
            active_batch=max(1,active_batch//2); print('OOM: retry batch',active_batch,flush=True); continue
        for sample,raw in zip(batch,raws):
            pred=parse_prediction(raw)
            append_jsonl(PRED,{'sample_id':sample['sample_id'],'model':MODEL_ID,'backend':f'official BF16 SDPA batch={len(batch)}','manifest_sha256':manifest['manifest_sha256'],'status':'ok' if pred else 'parse_failure','prediction':pred,'raw_response':raw,'latency_seconds':round(per_latency,4),'created_at':datetime.now(timezone.utc).isoformat()})
            completed+=1
        cursor+=len(batch)
        if completed%50<len(batch) or cursor==len(pending):
            elapsed=time.perf_counter()-started_all; rate=completed/max(elapsed,1e-6); eta=(len(pending)-cursor)/max(rate,1e-6)/3600
            print(f'[{cursor}/{len(pending)}] last={batch[-1]["sample_id"]} rate={rate:.3f} q/s ETA={eta:.2f}h',flush=True)
        if completed%100<len(batch): sync_mirror()
finally: sync_mirror()
print('inference complete')


In [ ]:
# Label join and full benchmark evaluation.
predictions=load_jsonl(PRED)
summary,scored=evaluate_records(manifest,predictions)
summary['run_config']=run_config
write_evaluation(OUT,summary,scored)
print(json.dumps(summary,ensure_ascii=False,indent=2,default=str))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
df=pd.DataFrame(scored)
fig,axes=plt.subplots(2,2,figsize=(16,11))
df.groupby('question_type')['correct'].mean().sort_values().mul(100).plot.barh(ax=axes[0,0],color='#76b900',title='Accuracy by capability')
df.groupby('source_dataset')['correct'].mean().sort_values().mul(100).plot.barh(ax=axes[0,1],color='#4c78a8',title='Accuracy by source')
df.groupby('category')['correct'].agg(['mean','count']).query('count>=20').sort_values('mean').tail(30)['mean'].mul(100).plot.barh(ax=axes[1,0],color='#f28e2b',title='Top category accuracy (n>=20)')
lat=df['latency_seconds'].dropna(); axes[1,1].hist(lat,bins=40,color='#59a14f'); axes[1,1].set_title('Per-question latency distribution'); axes[1,1].set_xlabel('seconds')
for ax in axes.flat: ax.grid(alpha=.2)
plt.tight_layout(); plt.savefig(OUT/'cosmos3_mmad_full_analysis.png',dpi=160,bbox_inches='tight'); plt.show()


In [ ]:
# Portable results only; model weights and MMAD images are excluded.
archive=shutil.make_archive(str(WORK/'cosmos3_nano_mmad_full_artifacts'),'zip',OUT)
print('DOWNLOAD:',archive,'size MiB:',round(Path(archive).stat().st_size/2**20,2))
